# House Stage 1: alternative generic-ballot predictors for Model A

Model A's national signal is currently the eve-of-election generic-ballot margin plus its 30-day change (momentum). This notebook compares four ways of entering the generic ballot, with everything else in Model A unchanged (`partisan_lean`, `inc_signed` fixed at +2.5, `partisan_lean:inc_signed`):

| spec | national predictor(s) | definition, with $m_d$ the daily generic-ballot margin (D − R) and $t$ the cutoff |
|---|---|---|
| **Current** | `gen_margin_raw` + `gb_momentum_30d_raw` | $m_t$ and $m_t - m_{t-30}$ |
| **GB-only** | `gen_margin_raw` | $m_t$ |
| **SMA30** | `gb_sma30_raw` | $\frac{1}{30}\sum_{k=0}^{29} m_{t-k}$ |
| **EWMA30** | `gb_ewma30_hl10_raw` | $\sum_{k=0}^{29} w_k m_{t-k} / \sum_k w_k$, with $w_k = 2^{-k/10}$ (fixed 10-day half-life) |

Each spec is evaluated alone (**A-…**) and in the fixed ensemble 0.85 A + 0.15 B with the existing lean-inclusive Model B (**AB-…**). **AB-Current** is `house_ensemble_ab_85_15`.

**Design:**

- **Fixed specifications:** the established configurations in `house.stage1` are unchanged. The alternative specs are defined in this notebook only. The half-life is fixed at 10 days, the blend weight at 0.15, and nothing is tuned.
- **Estimation:** equal-year WLS (`stage1.fit_predict`) on expanding windows from 2006 through the election before each eval year: 2018, 2020, 2022, and 2024.
- **Matched samples:** every model fits and scores the same district-years (asserted).
- **Cutoffs:** every predictor uses the same election-eve cutoff, the last daily value on or before the day before the election (`gb_date`). Each window holds only days on or before that cutoff (asserted below).
- **Development evidence:** this is a robustness comparison, not independent validation. 2024, like the other eval years, has already been examined.
- **Subsets:** valid targets, ex-ante competitive (|lean + eve-of-election generic ballot| ≤ 10), and ex-post competitive (|actual margin| ≤ 7.5).
- **Residuals:** actual − predicted, D − R points. *Pooled* weights every race equally, and *equal-year mean* averages the four years.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# The repository root is the first folder up from here that holds pyproject.toml, so this works at any depth
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import house.regression as hr
from house.experiments import evaluation, tracking
from house import stage1
from house.regression.data import ROOT
from house.regression.schema import KEY

with warnings.catch_warnings(action="ignore"):  # loading warns about known optional-column gaps
    predictors = hr.load_predictors()  # data/processed/house/modeling.csv, 2006-2026
    targets = hr.load_targets()        # data/processed/house/targets.csv, 2006-2024
daily = pd.read_csv(ROOT / "data/interim/generic_ballot_all.csv", parse_dates=["date"])  # one row per day, per cycle

## 1. The generic-ballot series and the smoothed predictors

**The source is already smoothed.** `data/interim/generic_ballot_all.csv` (built by `data-processing/house/01-generic-ballot.ipynb`) is FiveThirtyEight's daily generic-ballot *polling average*, not individual polls. The SMA and EWMA are therefore smoothings of an already-smoothed series. They mostly add lag, averaging the polling average over the final month, rather than removing poll-level noise.

**The modeling table's EWMA is not this spec.** `gb_margin_30d_ewma_raw` in modeling.csv is `ewm(span=30)` over the whole cycle: a half-life of about 10.4 days, using every earlier day of the cycle. The spec here is a 30-day window with an exact 10-day half-life, so both averages are computed below from the daily series. The SMA must reproduce `gb_margin_30d_mean_raw`, and that is asserted.

**Look-ahead checks:** for each election, the window is the 30 consecutive daily rows of its cycle ending at `gb_date`, the last day on or before the cutoff. The checks assert that:

- `gb_date` is on or before the cutoff;
- the window spans exactly 30 calendar days;
- the value at `gb_date` is `gen_margin_raw`;
- $m_t - m_{t-30}$ is `gb_momentum_30d_raw`.

In [ ]:
TARGET = stage1.TARGET
GB, MOMENTUM, SMA, EWMA = "gen_margin_raw", "gb_momentum_30d_raw", "gb_sma30_raw", "gb_ewma30_hl10_raw"
WINDOW_DAYS = 30
HALF_LIFE_DAYS = 10  # fixed, not tuned

cutoffs = predictors.groupby("year")[["gb_cutoff_date", "gb_date", GB, MOMENTUM, "gb_margin_30d_mean_raw",
                                      "gb_margin_30d_ewma_raw"]].first().dropna()
national = {}
for year, row in cutoffs.iterrows():
    cutoff, last = pd.Timestamp(row["gb_cutoff_date"]), pd.Timestamp(row["gb_date"])
    cycle = daily[(daily["election"] == year) & (daily["date"] <= cutoff)].set_index("date")["margin_raw"]
    window = cycle.loc[:last].iloc[-WINDOW_DAYS:]
    assert last <= cutoff and window.index[-1] == last, year
    assert (window.index[-1] - window.index[0]).days == WINDOW_DAYS - 1 and len(window) == WINDOW_DAYS, year
    age = (last - window.index).days.to_numpy()  # 0 at the cutoff, 29 for the oldest day
    weights = 0.5 ** (age / HALF_LIFE_DAYS)
    assert np.isclose(window.iloc[-1], row[GB]), year
    assert np.isclose(window.iloc[-1] - cycle.loc[last - pd.Timedelta(days=WINDOW_DAYS)], row[MOMENTUM]), year
    national[year] = {"cutoff": cutoff.date(), "gb_date": last.date(), GB: row[GB], MOMENTUM: row[MOMENTUM],
                      SMA: window.mean(), EWMA: (weights * window).sum() / weights.sum(),
                      "modeling.csv ewma (span 30, full cycle)": row["gb_margin_30d_ewma_raw"],
                      "mean |daily change|, final 30d": window.diff().abs().mean(),
                      "corr of successive daily changes": window.diff().autocorr()}
    assert np.isclose(national[year][SMA], row["gb_margin_30d_mean_raw"]), year  # reproduces the existing column
national = pd.DataFrame.from_dict(national, orient="index").rename_axis("year")
w = 0.5 ** (np.arange(WINDOW_DAYS) / HALF_LIFE_DAYS)
print(f"EWMA weight on the cutoff day: {w[0] / w.sum():.3f}; mean age: SMA {np.arange(WINDOW_DAYS).mean():.1f} days, "
      f"EWMA {(np.arange(WINDOW_DAYS) * w).sum() / w.sum():.1f} days")
national.style.format(precision=2)

In [ ]:
# The national series, one value per election, and how strongly the candidates correlate across 2006-2024
predictors = predictors.merge(national[[SMA, EWMA]], left_on="year", right_index=True, how="left")
display(Markdown("### Correlations of the national predictors across the ten elections"))
national[[GB, MOMENTUM, SMA, EWMA]].corr().style.format(precision=3)

## 2. Specifications and rolling fits

In [ ]:
EVAL_YEARS = [2018, 2020, 2022, 2024]
EVAL_RACE_TYPES = ["dr", "dr_multi", "dr_jungle", "dr_rcv", "dr_writein"]
EX_ANTE_THRESHOLD = 10
EX_POST_THRESHOLD = 7.5
COMPETITIVE = {
    "competitive_ex_ante": ("partisan_lean + gb_margin_2p_norm", EX_ANTE_THRESHOLD),  # known before the election
    "competitive_ex_post": ("margin_2p_norm", EX_POST_THRESHOLD),                     # the observed margin
}
SECTIONS = {"EX_ANTE": "competitive_ex_ante", "EX_POST": "competitive_ex_post", "ALL": "valid"}  # priority order
WEIGHT_B = 0.15

# Model A as it was when this comparison was made (2026-11-02, before production dropped momentum): the
# baseline here. Production adopted the result, so stage1.MODEL_A is now Model A without momentum.
MODEL_A_MOM = stage1.Component(
    "A_polling",
    f"{TARGET} ~ partisan_lean + gen_margin_raw + inc_signed + partisan_lean:inc_signed + gb_momentum_30d_raw",
    fixed=dict(stage1.MODEL_A.fixed),
)
FIXED = dict(stage1.MODEL_A.fixed)  # {"inc_signed": 2.5}


def model_a(name, national_terms):
    # Model A with its national polling terms replaced; everything else as in MODEL_A_MOM
    return stage1.Component(name, f"{TARGET} ~ partisan_lean + {' + '.join(national_terms)} + inc_signed"
                                  " + partisan_lean:inc_signed", fixed=FIXED)


NATIONAL_TERMS = {"Current": [GB, MOMENTUM], "GB-only": [GB], "SMA30": [SMA], "EWMA30": [EWMA]}
SPECS = {"Current": MODEL_A_MOM, "GB-only": model_a("A_gb", [GB]), "SMA30": model_a("A_sma30", [SMA]),
         "EWMA30": model_a("A_ewma30", [EWMA])}
assert set(SPECS["Current"].formula.split(" ~ ")[1].split(" + ")) == \
    set(model_a("check", [GB, MOMENTUM]).formula.split(" ~ ")[1].split(" + "))  # only the national terms differ
ENSEMBLES = {spec: stage1.Stage1Config(f"ab_{spec}", ((component, 1 - WEIGHT_B), (stage1.MODEL_B, WEIGHT_B)))
             for spec, component in SPECS.items()}
ALONE = {spec: stage1.Stage1Config(f"a_{spec}", ((component, 1.0),)) for spec, component in SPECS.items()}
for spec, component in SPECS.items():
    print(f"{spec:8s} {component.formula}   fixed: {dict(component.fixed)}")

In [ ]:
with warnings.catch_warnings(action="ignore"):
    rolling = {spec: stage1.rolling_predictions(config, EVAL_YEARS, predictors=predictors, targets=targets)
               for spec, config in ENSEMBLES.items()}

for year in EVAL_YEARS:
    runs = [r for spec in SPECS for r in rolling[spec][year].components.values()]
    assert len({(r.counts["train_rows_hash"], r.counts["eval_rows_hash"]) for r in runs}) == 1, year
    for spec in SPECS:
        assert rolling[spec][year].predictions[KEY].equals(rolling["Current"][year].predictions[KEY])
        assert np.array_equal(rolling[spec][year].predictions["prediction_B_fundamentals"],
                              rolling["Current"][year].predictions["prediction_B_fundamentals"])
    print(year, "trained on", runs[0].config["train_years"][0], "-", runs[0].config["train_years"][-1],
          "| train rows", runs[0].counts["n_train"], "| eval rows", len(rolling["Current"][year].predictions))

A_MODELS = [f"A-{spec}" for spec in SPECS]
AB_MODELS = [f"AB-{spec}" for spec in SPECS]
MODELS = A_MODELS + AB_MODELS
BASE = {m: ("A-Current" if m.startswith("A-") else "AB-Current") for m in MODELS if not m.endswith("Current")}
predictions = pd.concat([
    rolling["Current"][year].predictions[KEY].assign(**{
        **{f"A-{spec}": rolling[spec][year].predictions[f"prediction_{SPECS[spec].name}"] for spec in SPECS},
        **{f"AB-{spec}": rolling[spec][year].predictions["prediction"] for spec in SPECS},
    }) for year in EVAL_YEARS], ignore_index=True)

scores = {m: evaluation.evaluate_predictions(predictions[KEY].assign(prediction=predictions[m]), targets, predictors,
                                             TARGET, competitive=COMPETITIVE, race_types=EVAL_RACE_TYPES)
          for m in MODELS}
metrics = {m: s[0] for m, s in scores.items()}
scored = {m: s[2] for m, s in scores.items()}
assert all(scored[m][KEY].equals(scored["A-Current"][KEY]) for m in MODELS)  # identical valid district-years

races = scored["A-Current"][[*KEY, "actual_margin", "competitive_ex_ante", "competitive_ex_post"]]
PRED = pd.DataFrame({m: scored[m]["predicted_margin"].to_numpy() for m in MODELS}, index=races.index)
RESID = races["actual_margin"].to_numpy()[:, None] - PRED
MASKS = {"EX_ANTE": races["competitive_ex_ante"], "EX_POST": races["competitive_ex_post"],
         "ALL": pd.Series(True, index=races.index)}
scores["A-Current"][1]  # coverage: districts per eval year by exclusion reason, and the subset sizes

## 3. Performance

MAE and winner accuracy per subset, pooled, as the equal-year mean, and per eval year. Δ compares each alternative with the current specification at the same level: A-… against A-Current, and AB-… against AB-Current. ✓ marks an improvement and ✗ a regression.

In [ ]:
METRICS = ["n", "mae", "rmse", "accuracy_pct", "missed_races"]
DELTAS = ["mae", "accuracy_pct", "missed_races"]


def performance(period):
    def values(m, subset):
        if period == "pooled":
            return metrics[m].loc[(subset, "all")]
        if period == "equal-year mean":
            return sum(metrics[m].loc[(subset, y)] for y in EVAL_YEARS) / len(EVAL_YEARS)
        return metrics[m].loc[(subset, period)]

    rows = {}
    for m in MODELS:
        row = {}
        for section, subset in SECTIONS.items():
            v = values(m, subset)
            row |= {(section, k): v[k] for k in METRICS}
            if m in BASE:
                base = values(BASE[m], subset)
                row |= {(section, f"Δ{k}"): v[k] - base[k] for k in DELTAS}
        rows[m] = row
    return pd.DataFrame.from_dict(rows, orient="index")


for period in ["pooled", "equal-year mean"]:
    display(Markdown(f"### {period.capitalize()}, 2018–2024"))
    display(tracking.style_best_of(performance(period)))

In [ ]:
by_year = pd.concat({year: performance(year) for year in EVAL_YEARS}, names=["eval_year", "model"]).swaplevel()
by_year = by_year.reindex([(m, y) for m in MODELS for y in EVAL_YEARS])
display(Markdown("### Per eval year"))
tracking.style_best_of(by_year)

### Paired comparison with the current specification

On the same races: the mean change in absolute error (negative favors the alternative), its naive standard error (races treated as independent, which is optimistic because errors share a national component within each year), the share of races where the alternative is closer, and the per-year mean change.

In [ ]:
paired = {}
for m, base in BASE.items():
    diff = RESID[m].abs() - RESID[base].abs()
    for section, mask in MASKS.items():
        d = diff[mask]
        per_year = d.groupby(races.loc[mask, "year"]).mean()
        paired[(f"{m} vs {base}", section)] = {
            "n": len(d), "mean Δ|residual|": d.mean(), "naive se": d.std() / np.sqrt(len(d)),
            "% races closer": 100 * (d < 0).mean(), "years favoring alt": int((per_year < 0).sum()),
            **{f"Δ {y}": per_year[y] for y in EVAL_YEARS}}
pd.DataFrame.from_dict(paired, orient="index").rename_axis(["comparison", "subset"]).style.format(precision=3).format(
    {"n": "{:.0f}", "years favoring alt": "{:.0f}", "% races closer": "{:.1f}"})

## 4. Winner calls fixed and broken against the current specification

**Fixed:** the alternative is right and the current spec was wrong. **Broken:** the reverse. Net = fixed − broken, per subset. The list below the table shows every competitive race whose call an alternative changes.

In [ ]:
def correct(m):
    return pd.Series(np.sign(PRED[m]) == np.sign(races["actual_margin"]), index=races.index)


flip_rows = {}
for m, base in BASE.items():
    fixed, broken = correct(m) & ~correct(base), ~correct(m) & correct(base)
    for year in [*EVAL_YEARS, "pooled"]:
        row = {}
        for section, mask in MASKS.items():
            in_year = mask if year == "pooled" else mask & (races["year"] == year)
            f, b = int((fixed & in_year).sum()), int((broken & in_year).sum())
            row |= {(section, "fixed"): f, (section, "broken"): b, (section, "net"): f - b}
        flip_rows[(f"{m} vs {base}", year)] = row
flips = pd.DataFrame.from_dict(flip_rows, orient="index").rename_axis(["comparison", "eval_year"])
display(flips.style.map(lambda v: f"color: {tracking.GOOD}" if v > 0 else f"color: {tracking.BAD}" if v < 0 else "",
                        subset=[c for c in flips.columns if c[1] == "net"]))

competitive = races["competitive_ex_ante"] | races["competitive_ex_post"]
changed = []
for m, base in BASE.items():
    for i in races.index[competitive & (np.sign(PRED[m]) != np.sign(PRED[base]))]:
        r = races.loc[i]
        changed.append({"comparison": f"{m} vs {base}", "year": r["year"],
                        "district": evaluation.district_label(r["state"], r["district"]),
                        "ex_ante": r["competitive_ex_ante"], "ex_post": r["competitive_ex_post"],
                        "actual": r["actual_margin"], "current": PRED.loc[i, base], "alternative": PRED.loc[i, m],
                        "verdict": "fixed" if correct(m)[i] else "broken"})
changed = pd.DataFrame(changed, columns=["comparison", "year", "district", "ex_ante", "ex_post", "actual", "current",
                                         "alternative", "verdict"])
changed.style.format({"actual": "{:+.1f}", "current": "{:+.1f}", "alternative": "{:+.1f}"}).map(
    lambda v: f"color: {tracking.GOOD}" if v == "fixed" else f"color: {tracking.BAD}" if v == "broken" else "",
    subset=["verdict"])

## 5. Generic-ballot coefficients and how to read them

Each spec's national coefficients in each expanding window, with their spread. Every term is national, so the row-level standard errors (≈ 0.03–0.3) treat 2,000–3,500 races as independent evidence about what are really 6–9 election-level points. They overstate precision.

**The current spec as a filter on the daily series.** Momentum is $m_t - m_{t-30}$, so Current's national part, $\beta_G m_t + \beta_M (m_t - m_{t-30})$, is exactly $(\beta_G + \beta_M)\, m_t - \beta_M\, m_{t-30}$: a weight on the eve-of-election average and a weight on the average 30 days earlier. The second table shows those implied weights. By contrast, the SMA and EWMA put positive weights on every day of the final month.

In [ ]:
def coefficient_table(spec, column):
    name = SPECS[spec].name
    return pd.DataFrame({year: rolling[spec][year].components[name].coefficients[column] for year in EVAL_YEARS})


coefs = pd.concat({spec: coefficient_table(spec, "coef").loc[["Intercept", *NATIONAL_TERMS[spec], "partisan_lean"]]
                   for spec in SPECS}, names=["spec", "term"])
ses = pd.concat({spec: coefficient_table(spec, "std_err").loc[NATIONAL_TERMS[spec]] for spec in SPECS},
                names=["spec", "term"])
spread = pd.DataFrame({"mean": coefs.mean(axis=1), "sd": coefs.std(axis=1), "min": coefs.min(axis=1),
                       "max": coefs.max(axis=1)})
display(Markdown("### National coefficients by eval window (and partisan_lean for reference)"))
display(coefs.join(spread).style.format(precision=3))
display(Markdown("### Row-level standard errors of the national terms"))
display(ses.style.format(precision=3))

current = coefs.xs("Current", level="spec")
implied = pd.DataFrame({"weight on m_t (β_G + β_M)": current.loc[GB] + current.loc[MOMENTUM],
                        "weight on m_{t−30} (−β_M)": -current.loc[MOMENTUM],
                        "total (β_G)": current.loc[GB]}).T
display(Markdown("### Current spec: implied weights on the eve-of-election and 30-days-earlier averages"))
implied.style.format(precision=3)

## 6. Sensitivity to individual training elections

Each spec is refit on each window with one training year dropped, giving 6–9 refits per window. Two quantities are tracked:

- **Coefficient on the level:** the change in every prediction when the whole daily series shifts up by 1 point. That is $\beta_G$ for Current (a uniform shift leaves momentum unchanged) and the single national coefficient for the others, so it is comparable across specs.
- **Eval-year national prediction:** the mean prediction over the eval year's districts. Its range across the refits is how far one training election can move the national forecast.

In [ ]:
LEVEL = {"Current": GB, "GB-only": GB, "SMA30": SMA, "EWMA30": EWMA}
loyo = []
with warnings.catch_warnings(action="ignore"):
    for year in EVAL_YEARS:
        window = stage1.expanding_window(ALONE["Current"], year)
        for dropped in [None, *window]:
            years = [y for y in window if y != dropped]
            for spec, config in ALONE.items():
                result = stage1.fit_predict(config, years, year, predictors=predictors, targets=targets)
                coef = result.components[SPECS[spec].name].coefficients["coef"]
                loyo.append({"spec": spec, "eval window": year, "dropped": dropped or "none",
                             "level coefficient": coef[LEVEL[spec]], "momentum coefficient": coef.get(MOMENTUM),
                             "national prediction": result.predictions["prediction"].mean()})
loyo = pd.DataFrame(loyo)
refits = loyo[loyo["dropped"] != "none"]
full = loyo[loyo["dropped"] == "none"].set_index(["spec", "eval window"])
sensitivity = refits.groupby(["spec", "eval window"])[["level coefficient", "national prediction"]].agg(
    lambda s: s.max() - s.min()).add_suffix(": LOYO range")
sensitivity.insert(0, "level coefficient: full", full["level coefficient"])
sensitivity["national prediction: LOYO sd"] = refits.groupby(["spec", "eval window"])["national prediction"].std()
sensitivity = sensitivity.reindex([(s, y) for s in SPECS for y in EVAL_YEARS])
display(Markdown("### Leave-one-year-out sensitivity, per spec and window"))
display(sensitivity.style.format(precision=2))
display(Markdown("### Mean over the four windows"))
display(sensitivity.groupby(level="spec", sort=False).mean().style.format(precision=2))

display(Markdown("### Which dropped election moves the eval-year national prediction most (2024 window, "
                 "deviation from the full fit, points)"))
w24 = refits[refits["eval window"] == 2024].set_index(["dropped", "spec"])["national prediction"].unstack()
(w24 - full.xs(2024, level="eval window")["national prediction"]).reindex(columns=list(SPECS)).style.format(precision=2)

In [ ]:
COLORS = {"Current": "#2a78d6", "GB-only": "#eb6834", "SMA30": "#1baf7a", "EWMA30": "#eda100"}


def tidy(ax, title, xlabel, ylabel):
    ax.set_title(title, loc="left", fontsize=11)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", color="#e4e3df", linewidth=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(colors="#52514e")


fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
for offset, spec in enumerate(SPECS):
    x = np.array(EVAL_YEARS) + (offset - 1.5) * 0.3
    group = refits[refits["spec"] == spec].groupby("eval window")
    for ax, column in zip(axes, ["level coefficient", "national prediction"]):
        low, high = group[column].min(), group[column].max()
        centre = full.xs(spec, level="spec")[column]
        if column == "national prediction":  # deviation from the full fit, so the four specs share a scale
            low, high, centre = low - centre, high - centre, centre * 0
        ax.vlines(x, low.to_numpy(), high.to_numpy(), color=COLORS[spec], lw=2, alpha=0.6)
        ax.plot(x, centre.to_numpy(), "o", color=COLORS[spec], ms=8, label=spec)
for ax in axes:
    ax.set_xticks(EVAL_YEARS)
tidy(axes[0], "Coefficient on the generic-ballot level: full fit and LOYO range", "eval window", "coefficient")
tidy(axes[1], "Eval-year national prediction: LOYO range around the full fit", "eval window",
     "deviation from full fit (pts)")
axes[1].axhline(0, color="#8d8c86", lw=1)
axes[0].legend(frameon=False)
plt.tight_layout()

## 7. National directional error

- **Out of sample:** each eval year's mean residual, the error every district shares that year, per subset, with the pooled value, the equal-year mean, and the mean absolute yearly error.
- **In sample:** in the 2024 window (trained on 2006–2022), each training year's mean in-sample residual. That is how well each spec's national terms fit the elections they were estimated on.

In [ ]:
yearly_bias = pd.concat({section: RESID[mask].groupby(races.loc[mask, "year"]).mean()
                         for section, mask in MASKS.items()}, axis=1)
yearly_bias.loc["pooled"] = pd.Series({(s, m): RESID.loc[mask, m].mean() for s, mask in MASKS.items() for m in MODELS})
yearly_bias.loc["equal-year mean"] = yearly_bias.loc[EVAL_YEARS].mean()
yearly_bias.loc["mean |year bias|"] = yearly_bias.loc[EVAL_YEARS].abs().mean()
display(Markdown("### Out-of-sample mean residual by eval year"))
display(yearly_bias[[(s, m) for s in MASKS for m in A_MODELS]].style.format(precision=2))
display(yearly_bias[[(s, m) for s in MASKS for m in AB_MODELS]].style.format(precision=2))

in_sample = pd.DataFrame({spec: (lambda p: p[p["role"] == "train"].groupby("year")["residual"].mean())(
    rolling[spec][2024].components[SPECS[spec].name].predictions) for spec in SPECS})
in_sample.loc["sd across years"] = in_sample.std()
display(Markdown("### In-sample national residual by training year (2024 window, Model A alone)"))
in_sample.style.format(precision=2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, section in zip(axes, ["ALL", "EX_ANTE"]):
    for spec in SPECS:
        ax.plot(EVAL_YEARS, yearly_bias.loc[EVAL_YEARS, (section, f"AB-{spec}")], color=COLORS[spec], lw=2,
                marker="o", ms=8, label=f"AB-{spec}")
    ax.axhline(0, color="#8d8c86", lw=1)
    ax.set_xticks(EVAL_YEARS)
    tidy(ax, f"Ensemble national error by year ({section.lower()})", "eval year", "mean actual − predicted (pts)")
axes[0].legend(frameon=False)
plt.tight_layout()

## Conclusion

**Verdict:** the 30-day EWMA is more stable and easier to interpret than generic ballot + momentum, but it is **not a better alternative than the simplest one**, the eve-of-election margin alone (GB-only). It also gives up about as much competitive-race accuracy as GB-only does. It isn't worth adopting as specified.

**1. Smoothing adds little because the input is already smoothed.**

- **An average of an average:** the daily series is FiveThirtyEight's polling average, so the SMA and EWMA average it again over the final month. The EWMA puts 7.7% of its weight on the cutoff day, and its mean age is 9.6 days (14.5 for the SMA).
- **Nearly identical predictors:** across the ten elections, the SMA and EWMA correlate at 0.990–0.999 with the eve-of-election value. They differ materially only in years with a late move, such as 2010 (−6.6 at the cutoff vs −4.1 EWMA) and 2016.
- **Look-ahead checks pass:** every window ends at the same election-eve cutoff and uses only days on or before it. The SMA reproduces modeling.csv's `gb_margin_30d_mean_raw`. The modeling table's own EWMA column is a different construction (span 30 over the full cycle) and was not used.

**2. Accuracy: all three single-term specs trail Current by about 0.1 MAE.** Pooled MAE:

| spec | ex-ante (A) | ex-ante (AB) | ex-post (AB) | all (AB) | net AB calls vs Current |
|---|---|---|---|---|---|
| Current | 4.85 | 4.57 | 5.74 | 5.07 | — |
| GB-only | 4.99 | 4.68 | 5.87 | 5.17 | −3 |
| SMA30 | 4.97 | 4.66 | 5.85 | 5.14 | −3 |
| EWMA30 | 5.00 | 4.68 | 5.87 | 5.16 | −3 |

- **Where the loss is:** concentrated in 2018. In ex-ante races EWMA30 alone is +0.48 MAE that year, with −2 net ensemble calls. In 2022 and 2024 the EWMA is slightly *better* than Current in ex-ante races (−0.06 to −0.07 alone).
- **EWMA vs SMA:** EWMA30 is no better than SMA30 in any year. SMA30 is the best of the three, marginally (ensemble ex-ante +0.09).

**3. Stability and interpretability: EWMA beats Current, but not GB-only.**

- **Current's implied weights are hard to defend:** as a filter on the daily average, Current puts −0.37 on the eve-of-election value and +1.68 on the value 30 days earlier. Yet GB-only fits the training elections' national level better than either smoothed average, whose information is more lagged (in-sample national sd 2.33, vs 2.54 for SMA and 2.60 for EWMA). So "older polls are better" is not a consistent pattern in these data. Current's negative weight looks like a fit to a few elections (2008, 2012), not a smooth structure.
- **Coefficient on the level:** EWMA30's coefficient is 0.83–0.88 across windows, with a mean leave-one-year-out range of 0.33. That is much steadier than Current's 1.22–1.31 (range 0.86, plus a momentum coefficient that swings by 2–3 points). But GB-only is steadier still (0.78–0.82, range 0.25).
- **Sensitivity of the forecast itself:** when one training election is dropped, the eval-year national prediction moves over a mean range of 2.40 points for Current, 2.23 for EWMA30, 2.09 for SMA30, and 1.84 for GB-only. EWMA's steadier coefficient barely translates into a steadier forecast.
- **National direction:** EWMA's pooled bias is near zero (−0.05 vs −0.46 for Current). But its yearly national errors are larger (mean |year bias| 1.83 vs 1.36), led by 2018 (+2.64 vs +1.51).

**Recommendation.** Don't replace the current specification with the 30-day EWMA. Its cost in competitive-race accuracy is small (+0.11 to +0.14 ex-ante MAE, −3 ensemble calls, mostly 2018) but the same as GB-only's. Meanwhile:

- **GB-only matches its accuracy and does better on every other criterion:** fewer moving parts, no smoothing parameter, the steadiest coefficient, and the least sensitive national forecast.
- **Smoothing an existing polling average adds lag without adding information.**

The real choice is therefore Current vs GB-only, as in `01_reg_model_a_no_momentum.ipynb`. On these four elections, Current keeps a small, 2018-driven accuracy edge, and GB-only is the more stable and parsimonious fallback.

All eval years, 2024 included, have been examined before, so this is a robustness comparison, not validation.